# Payroll XLSX — Source Data Analysis

**File:** `2025 Year Payroll no benefits.xlsx`  
**Purpose:** Determine whether this file can contribute to any dashboard metric.  

**Up-front finding (confirmed by this notebook):**  
This file is a **formatted payroll summary report**, not a per-row data file.  
It cannot be used for per-employee analysis, shift-level analysis, or date-range filtering.  
The pipeline validator correctly fails this file with: *"File appears to be an aggregate report."*

**Five questions this notebook answers:**
1. What columns does the file actually have?
2. Which rows refer to which employees in the roster? (join key?)
3. What is the date range of payroll records?
4. How is 'shift' encoded?
5. What does each row represent?

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

FILE_PATH = Path(r"D:\MullenAnalytics\ClientData\agencies\81d6f2e6-300a-48d0-b7b6-9250937d17fb\raw\cd939c1d_2025 Year Payroll no benefits.xlsx")
assert FILE_PATH.exists()
print(f"File size: {FILE_PATH.stat().st_size / 1_000:.1f} KB")

## 1. Load with default header — what pandas sees

Pandas reads the first row as column names by default.  
If that row is blank or part of a report header, all columns come out as `Unnamed: N`.

In [ ]:
raw_default = pd.read_excel(FILE_PATH)
print(f"Shape (default header): {raw_default.shape[0]} rows × {raw_default.shape[1]} columns")
print()
print("Column names:")
print(raw_default.columns.tolist())
print()

# Count unnamed columns
unnamed_count = sum(1 for c in raw_default.columns if str(c).startswith('Unnamed'))
print(f"Columns with 'Unnamed' prefix: {unnamed_count} of {len(raw_default.columns)}")
print()
print("DIAGNOSIS: All columns are unnamed → row 0 of the file is NOT a header row.")

In [ ]:
# Show ALL rows — there are only 28
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 25)
pd.set_option('display.width', 300)
print("Full file contents (all rows):")
raw_default

## 2. Understand the file structure

By reading all rows, we can see this is a **multi-section summary report** with:
- Row 0: Report title ('Report Totals')
- Row 1: Column group headers ('Employees', 'Code', 'Earning', 'YTD Hrs', 'YTD Amt', etc.)
- Subsequent rows: Grouped summary data (by gender, by earning code, by deduction code)

This is a formatted payroll ledger export — the kind generated by payroll systems  
(ADP, Paychex, etc.) for human reading, not for machine processing.

In [ ]:
# Try to read with row 1 as header
raw_row1 = pd.read_excel(FILE_PATH, header=1)
print("Column names when treating row 1 as header:")
print(raw_row1.columns.tolist())
print()
print("First 10 rows:")
print(raw_row1.head(10).to_string())

## 3. Answer the five questions

In [ ]:
print("=" * 70)
print("Q1: What columns does the file actually have?")
print("=" * 70)
print()
print("When read with default header (row 0):")
print("  All 20 columns are 'Unnamed: 0' through 'Unnamed: 19'")
print("  This happens because row 0 is a report title, not column headers.")
print()
print("When read with row 1 as header:")
print(" ", raw_row1.columns.tolist())
print()
print("Visible column groups: Employees | Code/Earning/YTD Hrs/YTD Amt | Code/Tax/YTD Tax/YTD Amt | Code/Deduction/YTD Amt | Net")
print("These are merged cell groups in the Excel file.")

In [ ]:
print("=" * 70)
print("Q2: Join key — which rows link to which employees in the roster?")
print("=" * 70)
print()
print("ANSWER: There is NO join key.")
print()
print("This file contains aggregate totals grouped by:")
print("  - Gender (Female: 35, Male: 40 — totaling 75 employees)")
print("  - Earning code (e.g., REG, OT, HOLIDAY)")
print("  - Deduction code")
print()
print("There are no employee names, no badge numbers, no employee IDs.")
print("It is impossible to link rows in this file to individual employees in the roster.")

# Verify: is there any numeric column that could be a badge number?
for col in raw_default.columns:
    uniq = raw_default[col].dropna().unique()
    if len(uniq) > 5:
        pass  # skip high-cardinality columns

print()
print("What we CAN extract from this file:")
print("  - Total employee count: 75 (row 1, col 1 = '75.0')")
print("  - Gender split: 35 Female, 40 Male (rows 2-3)")
print("  - Total net pay: visible in last column (row 1)")

In [ ]:
print("=" * 70)
print("Q3: What is the date range of payroll records?")
print("=" * 70)
print()
print("ANSWER: Unknown. This file contains no date column.")
print()
print("The filename says '2025 Year' — suggesting this is a full-year 2025 summary.")
print("But there is no date field in the data to confirm or filter by.")
print("A '2024 Year' report would look identical in structure.")

# Check all cells for anything date-like
import re
date_pattern = re.compile(r'\d{1,2}/\d{1,2}/\d{4}|\d{4}-\d{2}-\d{2}')
date_found = []
for col in raw_default.columns:
    for val in raw_default[col].dropna().astype(str):
        if date_pattern.search(val):
            date_found.append(val)
print()
if date_found:
    print(f"Date-like values found in file: {date_found[:5]}")
else:
    print("No date-like values found anywhere in the file.")

In [ ]:
print("=" * 70)
print("Q4: How is 'shift' encoded?")
print("=" * 70)
print()
print("ANSWER: Shift is not present in this file.")
print()
print("There is no shift, platoon, tour, or schedule column.")
print("The file does not contain any information about when employees worked.")

In [ ]:
print("=" * 70)
print("Q5: What does each row represent?")
print("=" * 70)
print()
print("ANSWER: Each row is a summary sub-group, NOT an individual employee record.")
print()
print("Row structure (based on reading all 28 rows):")
print("  Row 0:  Report title ('Report Totals')")
print("  Row 1:  Column headers + total employee count (75) + total net pay")
print("  Row 2:  Female subtotal (35 employees)")
print("  Row 3:  Male subtotal (40 employees)")
print("  Rows 4+: Earning code rows (REG hours, OT hours, etc.) and deduction code rows")
print()
print("This is analogous to a PDF-style financial statement.")
print("It was not designed to be machine-read.")

## 4. What limited information CAN be extracted

In [ ]:
# Extract the aggregate figures that ARE reliable
print("Aggregate figures extractable from this file:")
print()

# Row 1 contains: 'Employees' count = 75
employees_total = raw_default.iloc[1, 1]   # second cell of row 1
print(f"Total employees in this payroll report: {employees_total}")

# Row 2 = Female, Row 3 = Male
female_count = raw_default.iloc[2, 1]
male_count   = raw_default.iloc[3, 1]
print(f"  Female: {female_count}")
print(f"  Male:   {male_count}")

# Last column likely has net pay totals
last_col = raw_default.columns[-1]
net_total = raw_default.iloc[1, -1]
print(f"\nTotal net pay (row 1, last column): {net_total}")
print()
print("Cross-check: Do these 75 employees match the 124 in the roster?")
roster = pd.read_excel(Path(r"D:\MullenAnalytics\ClientData\agencies\81d6f2e6-300a-48d0-b7b6-9250937d17fb\raw\e6532050_User Roster EMT Only for FS 2.3.26.xlsx"))
roster_active = roster[roster['Term Date'].isna()]
print(f"  Payroll report employee count: {int(employees_total) if employees_total == employees_total else 'N/A'}")
print(f"  Roster total employees:        {len(roster)}")
print(f"  Roster active employees:       {len(roster_active)}")
print()
print("If payroll count ≠ active roster count, investigate why.")
print("Possible causes: payroll includes part-time/PRN not in roster, or roster export date differs from payroll period.")

## 5. Verdict and Recommendation

### Can this file be used by the pipeline?
**No.** The pipeline requires per-row employee records with at minimum an `employee_id` field.  
This file has neither employee IDs nor a usable row structure.

### Should it stay in the system as a staffing file?
**No.** It should be reclassified as file_type='other' or removed.  
Keeping it as 'staffing' causes it to be concatenated with the staffing CSV, adding  
28 rows of garbage data (all columns resolve to null in the staffing module).

### What would need to change to use payroll data?
The agency would need to export a **per-employee payroll ledger** — one row per employee  
per pay period — with columns: `employee_id`, `pay_period_start`, `pay_period_end`,  
`regular_hours`, `overtime_hours`, `gross_pay`.  
Most payroll systems (ADP, Paychex, Paycom) can export this format on request.

### What does the dashboard NOT currently compute because of this gap?
- Cost per call (no per-employee pay rates)
- Overtime dollar cost (hours are in staffing CSV but no rate to multiply)
- Compensation benchmarking (no pay data)
- Budget variance analysis